##                   **Notebook1- Setup and Initial Exploration**


##### Football Match Prediction
##### Name: Vishal Chaudhary
##### Student Number: X23332794

###  **1. Environment Setup**

In [6]:
import os
project_name = "DOMAIN_APPLICATIONS_PROJECT"

# Define subdirectories
folders = [
    "data/raw",
    "data/processed",
    "data/features",
    "data/predictions",
    "docs",
    "models",
    "notebooks",
    "results/figures",
    "results/metrics",
    "results/reports",
    "src"
]

# Create all directories
for folder in folders:
    path = os.path.join(project_name, folder)
    os.makedirs(path, exist_ok=True)

print(f"Project folder structure '{project_name}' created successfully!")


Project folder structure 'DOMAIN_APPLICATIONS_PROJECT' created successfully!


###  **2. Libraries Import**

In [7]:
import time
import json
import requests
import pandas as pd
import numpy as np
from datetime import datetime

###  **3. API Setup**

In [8]:
API_KEY = "b45dd66d18aef1ffb86f5f1df8c83aed"

# Check API Status
print("TEST 1: API Status")
url = "https://v3.football.api-sports.io/status"
headers = {"x-apisports-key": API_KEY}
response = requests.get(url, headers=headers)
print(response.json())

TEST 1: API Status
{'get': 'status', 'parameters': [], 'errors': [], 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': {'account': {'firstname': 'vishal', 'lastname': 'chaudhary', 'email': 'ch.vishal400@gmail.com'}, 'subscription': {'plan': 'Pro', 'end': '2025-11-17T18:37:38+00:00', 'active': True}, 'requests': {'current': 0, 'limit_day': 7500}}}


In [9]:
# Get Sample Fixtures
print(" Sample Fixtures (Premier League)")
url = "https://v3.football.api-sports.io/fixtures"
params = {"league": 39, "season": 2023, "last": 3}
response = requests.get(url, headers=headers, params=params)
data = response.json()
print(f"Total matches: {len(data['response'])}")
for match in data['response'][:3]:
    home = match['teams']['home']['name']
    away = match['teams']['away']['name']
    score = f"{match['goals']['home']}-{match['goals']['away']}"
    print(f"  {home} {score} {away}")

print("\n API is working.")

 Sample Fixtures (Premier League)
Total matches: 3
  Luton 2-4 Fulham
  Liverpool 2-0 Wolves
  Arsenal 2-1 Everton

 API is working.


###  **4. API Configuration**

In [10]:
# API Configuration
API_KEY = "b45dd66d18aef1ffb86f5f1df8c83aed"
BASE_URL = "https://v3.football.api-sports.io"

headers = {
    'x-rapidapi-key': API_KEY,
    'x-rapidapi-host': 'v3.football.api-sports.io'}

# Major leagues to focus on 
LEAGUES = {
    'Premier League': 39,
    'La Liga': 140,
    'Serie A': 135,
    'Bundesliga': 78,
    'Ligue 1': 61,
    'UEFA Champions League': 2,
    'UEFA Europa League': 3,
    'UEFA Europa Conference League': 848,
    'Eredivisie': 88,
    'Primeira Liga': 94}

###  **5. Path To Save Raw Data**

In [11]:
# Define the path to save raw data
RAW_DATA_PATH = os.path.join("Users","vishalchaudhary","Desktop","Football_Project","DOMAIN_APPLICATIONS_PROJECT", "data", "raw")
os.makedirs(RAW_DATA_PATH, exist_ok=True)

def save_to_raw_data(filename, data_df):
    """
    Save a DataFrame to the raw data directory
    
    Parameters:
    filename : str
        Name of the file (will be prefixed with timestamp)
    data_df : pandas.DataFrame
        DataFrame to save
    
    Returns:
    str : Full path of the saved file
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    full_filename = f"{filename}_{timestamp}.csv"
    full_path = os.path.join(RAW_DATA_PATH, full_filename)
    data_df.to_csv(full_path, index=False)
    print(f"Saved to: {full_path}")
    return full_path

###  **6. Retriveing Data Via API**

####  **6.1 Get Fixtures by Season**

In [13]:
def get_fixtures_by_season(league_id, season):
    """Fetch all fixtures for a specific league and season"""
    url = f"{BASE_URL}/fixtures"
    params = {
        'league': league_id,
        'season': season
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        
        print(f"API Response: {data['response'][:2] if data['response'] else 'No data'}")
        return data['response']
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        return []

def process_fixture_data(fixtures):
    """Process raw fixture data into structured format"""
    processed_data = []
    
    for fixture in fixtures:
        # Only include finished matches
        if fixture['fixture']['status']['short'] not in ['FT', 'AET', 'PEN']:
            continue
            
        match_data = {
            'fixture_id': fixture['fixture']['id'],
            'date': fixture['fixture']['date'],
            'timestamp': fixture['fixture']['timestamp'],
            'league_id': fixture['league']['id'],
            'league_name': fixture['league']['name'],
            'season': fixture['league']['season'],
            'round': fixture['league']['round'],
            'home_team_id': fixture['teams']['home']['id'],
            'home_team_name': fixture['teams']['home']['name'],
            'away_team_id': fixture['teams']['away']['id'],
            'away_team_name': fixture['teams']['away']['name'],
            'home_goals': fixture['goals']['home'],
            'away_goals': fixture['goals']['away'],
            'home_score_halftime': fixture['score']['halftime']['home'],
            'away_score_halftime': fixture['score']['halftime']['away'],
            'home_score_fulltime': fixture['score']['fulltime']['home'],
            'away_score_fulltime': fixture['score']['fulltime']['away'],
            'venue_name': fixture['fixture']['venue']['name'],
            'venue_city': fixture['fixture']['venue']['city'],
            'referee': fixture['fixture']['referee'],
            'status': fixture['fixture']['status']['short']
        }
        processed_data.append(match_data)
    
    return processed_data

def main():
    """Main function to download historical fixtures from 2018 to 2023"""
    all_fixtures = []
    seasons = list(range(2018, 2024))  # 2018 to 2023
    
    print("Starting Historical Fixtures Download (2018-2023)")
    
    for league_name, league_id in LEAGUES.items():
        print(f"\nProcessing {league_name} (ID: {league_id})")
        
        for season in seasons:
            print(f"  Fetching season {season}...", end=" ")
            fixtures = get_fixtures_by_season(league_id, season)
            
            if fixtures:
                processed = process_fixture_data(fixtures)
                all_fixtures.extend(processed)
                print(f" Got {len(processed)} matches")
            else:
                print("✗No data")
            
            # Rate limiting - be respectful to API
            time.sleep(2)  # 2 seconds between requests
    
    # Save to CSV
    if all_fixtures:
        df = pd.DataFrame(all_fixtures)
        saved_file_path = save_to_raw_data("historical_fixtures_2018_2023", df)
        df.to_csv(saved_file_path, index=False)
        
        print(f"SUCCESS! Downloaded {len(df)} total matches")
        print(f"Saved to: {saved_file_path}")
        print("\nData Summary:")
        print(f"  Leagues: {df['league_name'].nunique()}")
        print(f"  Seasons: {df['season'].nunique()}")
        print(f"  Teams: {df['home_team_name'].nunique()}")
        print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
    else:
        print("\n No data collected!")

if __name__ == "__main__":
    main()

Starting Historical Fixtures Download (2018-2023)

Processing Premier League (ID: 39)
  Fetching season 2018... API Response: [{'fixture': {'id': 65, 'referee': 'Andre Marriner, England', 'timezone': 'UTC', 'date': '2018-08-10T19:00:00+00:00', 'timestamp': 1533927600, 'periods': {'first': 1533927600, 'second': 1533931200}, 'venue': {'id': 556, 'name': 'Old Trafford', 'city': 'Manchester'}, 'status': {'long': 'Match Finished', 'short': 'FT', 'elapsed': 90, 'extra': None}}, 'league': {'id': 39, 'name': 'Premier League', 'country': 'England', 'logo': 'https://media.api-sports.io/football/leagues/39.png', 'flag': 'https://media.api-sports.io/flags/gb-eng.svg', 'season': 2018, 'round': 'Regular Season - 1', 'standings': True}, 'teams': {'home': {'id': 33, 'name': 'Manchester United', 'logo': 'https://media.api-sports.io/football/teams/33.png', 'winner': True}, 'away': {'id': 46, 'name': 'Leicester', 'logo': 'https://media.api-sports.io/football/teams/46.png', 'winner': False}}, 'goals': {'h

####  **6.2 Get Teams In League**

In [15]:
def get_teams_in_league(league_id, season):
    """Get all teams in a specific league and season"""
    url = f"{BASE_URL}/teams"
    params = {
        'league': league_id,
        'season': season
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        return data['response']
    except requests.exceptions.RequestException as e:
        print(f"Error fetching teams: {e}")
        return []

def get_team_statistics(team_id, league_id, season):
    """Fetch detailed statistics for a specific team"""
    url = f"{BASE_URL}/teams/statistics"
    params = {
        'team': team_id,
        'league': league_id,
        'season': season
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        return data['response']
    except requests.exceptions.RequestException as e:
        print(f"Error fetching team stats for team {team_id}: {e}")
        return None

def process_team_statistics(stats, team_id, team_name, league_id, league_name, season):
    """Process raw team statistics into structured format"""
    if not stats:
        return None
    
    fixtures = stats.get('fixtures', {})
    goals = stats.get('goals', {})
    
    team_data = {
        'team_id': team_id,
        'team_name': team_name,
        'league_id': league_id,
        'league_name': league_name,
        'season': season,
        
        # Fixtures played
        'matches_played_home': fixtures.get('played', {}).get('home', 0),
        'matches_played_away': fixtures.get('played', {}).get('away', 0),
        'matches_played_total': fixtures.get('played', {}).get('total', 0),
        
        # Wins
        'wins_home': fixtures.get('wins', {}).get('home', 0),
        'wins_away': fixtures.get('wins', {}).get('away', 0),
        'wins_total': fixtures.get('wins', {}).get('total', 0),
        
        # Draws
        'draws_home': fixtures.get('draws', {}).get('home', 0),
        'draws_away': fixtures.get('draws', {}).get('away', 0),
        'draws_total': fixtures.get('draws', {}).get('total', 0),
        
        # Losses
        'losses_home': fixtures.get('loses', {}).get('home', 0),
        'losses_away': fixtures.get('loses', {}).get('away', 0),
        'losses_total': fixtures.get('loses', {}).get('total', 0),
        
        # Goals For
        'goals_for_home': goals.get('for', {}).get('total', {}).get('home', 0),
        'goals_for_away': goals.get('for', {}).get('total', {}).get('away', 0),
        'goals_for_total': goals.get('for', {}).get('total', {}).get('total', 0),
        'goals_for_avg_home': goals.get('for', {}).get('average', {}).get('home', 0),
        'goals_for_avg_away': goals.get('for', {}).get('average', {}).get('away', 0),
        'goals_for_avg_total': goals.get('for', {}).get('average', {}).get('total', 0),
        
        # Goals Against
        'goals_against_home': goals.get('against', {}).get('total', {}).get('home', 0),
        'goals_against_away': goals.get('against', {}).get('total', {}).get('away', 0),
        'goals_against_total': goals.get('against', {}).get('total', {}).get('total', 0),
        'goals_against_avg_home': goals.get('against', {}).get('average', {}).get('home', 0),
        'goals_against_avg_away': goals.get('against', {}).get('average', {}).get('away', 0),
        'goals_against_avg_total': goals.get('against', {}).get('average', {}).get('total', 0),
        
        # Clean sheets
        'clean_sheets_home': stats.get('clean_sheet', {}).get('home', 0),
        'clean_sheets_away': stats.get('clean_sheet', {}).get('away', 0),
        'clean_sheets_total': stats.get('clean_sheet', {}).get('total', 0),
        
        # Failed to score
        'failed_to_score_home': stats.get('failed_to_score', {}).get('home', 0),
        'failed_to_score_away': stats.get('failed_to_score', {}).get('away', 0),
        'failed_to_score_total': stats.get('failed_to_score', {}).get('total', 0),
        
        # Form
        'form': stats.get('form', ''),
        
        # Biggest stats
        'biggest_streak_wins': stats.get('biggest', {}).get('streak', {}).get('wins', 0),
        'biggest_streak_draws': stats.get('biggest', {}).get('streak', {}).get('draws', 0),
        'biggest_streak_loses': stats.get('biggest', {}).get('streak', {}).get('loses', 0),
        'biggest_wins_home': stats.get('biggest', {}).get('wins', {}).get('home', ''),
        'biggest_wins_away': stats.get('biggest', {}).get('wins', {}).get('away', ''),
        'biggest_loses_home': stats.get('biggest', {}).get('loses', {}).get('home', ''),
        'biggest_loses_away': stats.get('biggest', {}).get('loses', {}).get('away', ''),
    }
    
    return team_data

def main():
    """Main function to download team statistics from 2018 to 2023"""
    all_team_stats = []
    seasons = list(range(2018, 2024))  # 2018 to 2023
    
    print("Starting Team Statistics Download (2018-2023)")
    
    for league_name, league_id in LEAGUES.items():
        print(f"\n{league_name} (ID: {league_id})")
        
        for season in seasons:
            print(f"  Season {season}:")
            
            # Get teams for this season
            teams = get_teams_in_league(league_id, season)
            print(f"    Found {len(teams)} teams")
            
            # Get statistics for each team
            for i, team_data in enumerate(teams, 1):
                team_id = team_data['team']['id']
                team_name = team_data['team']['name']
                
                print(f"    [{i}/{len(teams)}] {team_name}...", end=" ")
                
                stats = get_team_statistics(team_id, league_id, season)
                
                if stats:
                    processed = process_team_statistics(
                        stats, team_id, team_name, league_id, league_name, season
                    )
                    if processed:
                        all_team_stats.append(processed)
                        print("")
                    else:
                        print("(processing failed)")
                else:
                    print(" (no data)")
                
                # Rate limiting
                time.sleep(1.5)
            
            # Extra delay between seasons
            time.sleep(2)
    
    # Save to CSV
    if all_team_stats:
        df = pd.DataFrame(all_team_stats)
        saved_file_path = save_to_raw_data("league_teams", df)
        df.to_csv(saved_file_path, index=False)
        
        print(f"SUCCESS- Downloaded statistics for {len(df)} team-seasons")
        print(f"Saved to: {saved_file_path}")
        print("\nData Summary:")
        print(f"  Leagues: {df['league_name'].nunique()}")
        print(f"  Unique teams: {df['team_name'].nunique()}")
        print(f"  Seasons covered: {sorted(df['season'].unique())}")
    else:
        print("\n No data collected!")

if __name__ == "__main__":
    main()

Starting Team Statistics Download (2018-2023)

Premier League (ID: 39)
  Season 2018:
    Found 20 teams
    [1/20] Manchester United... 
    [2/20] Newcastle... 
    [3/20] Bournemouth... 
    [4/20] Fulham... 
    [5/20] Huddersfield... 
    [6/20] Watford... 
    [7/20] Wolves... 
    [8/20] Liverpool... 
    [9/20] Southampton... 
    [10/20] Arsenal... 
    [11/20] Cardiff... 
    [12/20] Burnley... 
    [13/20] Everton... 
    [14/20] Leicester... 
    [15/20] Tottenham... 
    [16/20] West Ham... 
    [17/20] Chelsea... 
    [18/20] Manchester City... 
    [19/20] Brighton... 
    [20/20] Crystal Palace... 
  Season 2019:
    Found 20 teams
    [1/20] Manchester United... 
    [2/20] Newcastle... 
    [3/20] Bournemouth... 
    [4/20] Watford... 
    [5/20] Wolves... 
    [6/20] Liverpool... 
    [7/20] Southampton... 
    [8/20] Arsenal... 
    [9/20] Burnley... 
    [10/20] Everton... 
    [11/20] Leicester... 
    [12/20] Tottenham... 
    [13/20] West Ham... 
    [14/20] Che

####  **6.3 Get League Standings**

In [16]:
def get_standings(league_id, season):
    """Fetch league standings for a specific season"""
    url = f"{BASE_URL}/standings"
    params = {
        'league': league_id,
        'season': season
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        return data['response']
    except requests.exceptions.RequestException as e:
        print(f"Error fetching standings: {e}")
        return []

def process_standings_data(standings_response, league_id, league_name, season):
    """Process raw standings data into structured format"""
    processed_data = []
    
    if not standings_response:
        return processed_data
    
    # The API returns standings in a nested structure
    for standing_group in standings_response:
        league_info = standing_group.get('league', {})
        standings = league_info.get('standings', [[]])[0]  # First group is main standings
        
        for team_standing in standings:
            team_data = {
                # Basic info
                'season': season,
                'league_id': league_id,
                'league_name': league_name,
                'team_id': team_standing['team']['id'],
                'team_name': team_standing['team']['name'],
                
                # Position and form
                'rank': team_standing['rank'],
                'points': team_standing['points'],
                'goal_diff': team_standing['goalsDiff'],
                'group': team_standing.get('group', 'League'),
                'form': team_standing.get('form', ''),
                'status': team_standing.get('status', ''),
                'description': team_standing.get('description', ''),
                
                # Overall stats
                'matches_played': team_standing['all']['played'],
                'wins': team_standing['all']['win'],
                'draws': team_standing['all']['draw'],
                'losses': team_standing['all']['lose'],
                'goals_for': team_standing['all']['goals']['for'],
                'goals_against': team_standing['all']['goals']['against'],
                
                # Home stats
                'matches_played_home': team_standing['home']['played'],
                'wins_home': team_standing['home']['win'],
                'draws_home': team_standing['home']['draw'],
                'losses_home': team_standing['home']['lose'],
                'goals_for_home': team_standing['home']['goals']['for'],
                'goals_against_home': team_standing['home']['goals']['against'],
                
                # Away stats
                'matches_played_away': team_standing['away']['played'],
                'wins_away': team_standing['away']['win'],
                'draws_away': team_standing['away']['draw'],
                'losses_away': team_standing['away']['lose'],
                'goals_for_away': team_standing['away']['goals']['for'],
                'goals_against_away': team_standing['away']['goals']['against'],
                
                # Update timestamp
                'update_date': team_standing.get('update', '')
            }
            
            # Calculate additional metrics
            if team_data['matches_played'] > 0:
                team_data['points_per_game'] = round(team_data['points'] / team_data['matches_played'], 2)
                team_data['win_percentage'] = round(team_data['wins'] / team_data['matches_played'] * 100, 2)
                team_data['goals_per_game'] = round(team_data['goals_for'] / team_data['matches_played'], 2)
                team_data['goals_conceded_per_game'] = round(team_data['goals_against'] / team_data['matches_played'], 2)
            else:
                team_data['points_per_game'] = 0
                team_data['win_percentage'] = 0
                team_data['goals_per_game'] = 0
                team_data['goals_conceded_per_game'] = 0
            
            processed_data.append(team_data)
    
    return processed_data

def get_standings_by_round(league_id, season):
    """Get standings at different points in the season (by round)"""
    # This function can be used to track standings evolution throughout the season
    # For now, we'll just get final standings, but this could be extended
    pass

def main():
    """Main function to download league standings from 2018 to 2023"""
    all_standings = []
    seasons = list(range(2018, 2024))  # 2018 to 2023
    
    print("Starting League Standings Download (2018-2023)")
    
    for league_name, league_id in LEAGUES.items():
        print(f"\n{league_name} (ID: {league_id})")
        
        for season in seasons:
            print(f"  Season {season}...", end=" ")
            
            standings_response = get_standings(league_id, season)
            
            if standings_response:
                processed = process_standings_data(standings_response, league_id, league_name, season)
                all_standings.extend(processed)
                print(f" Got {len(processed)} teams")
            else:
                print("No data")
            
            # Rate limiting
            time.sleep(0.4)
    
    # Save to CSV
    if all_standings:
        df = pd.DataFrame(all_standings)
        saved_file_path = save_to_raw_data("league_standings", df)
        df.to_csv(saved_file_path, index=False)
        
        print(f"SUCCESS Downloaded standings for {len(df)} team-seasons")
        print(f"Saved to: {saved_file_path}")
        print("\nData Summary:")
        print(f"  Leagues: {df['league_name'].nunique()}")
        print(f"  Seasons: {sorted(df['season'].unique())}")
        print(f"  Unique teams: {df['team_name'].nunique()}")
        print(f"  Total records: {len(df)}")
    else:
        print("\n No data collected!")

if __name__ == "__main__":
    main()

Starting League Standings Download (2018-2023)

Premier League (ID: 39)
  Season 2018...  Got 20 teams
  Season 2019...  Got 20 teams
  Season 2020...  Got 20 teams
  Season 2021...  Got 20 teams
  Season 2022...  Got 20 teams
  Season 2023...  Got 20 teams

La Liga (ID: 140)
  Season 2018...  Got 20 teams
  Season 2019...  Got 20 teams
  Season 2020...  Got 20 teams
  Season 2021...  Got 20 teams
  Season 2022...  Got 20 teams
  Season 2023...  Got 20 teams

Serie A (ID: 135)
  Season 2018...  Got 20 teams
  Season 2019...  Got 20 teams
  Season 2020...  Got 20 teams
  Season 2021...  Got 20 teams
  Season 2022...  Got 20 teams
  Season 2023...  Got 20 teams

Bundesliga (ID: 78)
  Season 2018...  Got 18 teams
  Season 2019...  Got 18 teams
  Season 2020...  Got 18 teams
  Season 2021...  Got 18 teams
  Season 2022...  Got 18 teams
  Season 2023...  Got 18 teams

Ligue 1 (ID: 61)
  Season 2018...  Got 20 teams
  Season 2019...  Got 20 teams
  Season 2020...  Got 20 teams
  Season 2021.

####  **6.4 Get Player Statistics Basic In Leagues**

In [56]:
RAW_DATA_PATH = os.path.join("DOMAIN_APPLICATIONS_PROJECT", "data", "raw")
os.makedirs(RAW_DATA_PATH, exist_ok=True)

def save_to_raw_data(filename, data_df):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    full_filename = f"{filename}_{timestamp}.csv"
    full_path = os.path.join(RAW_DATA_PATH, full_filename)
    data_df.to_csv(full_path, index=False)
    print(f"✅ Saved to: {full_path}")
    return full_path

def get_teams_in_league(league_id, season):
    url = f"{BASE_URL}/teams"
    params = {'league': league_id, 'season': season}
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        return [team['team'] for team in data['response']]
    except requests.exceptions.RequestException as e:
        print(f"Error fetching teams: {e}")
        return []

def get_squad(team_id):
    url = f"{BASE_URL}/players/squads"
    params = {'team': team_id}
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        return data['response']
    except requests.exceptions.RequestException as e:
        print(f"Error fetching squad for team {team_id}: {e}")
        return []

def process_squad_data(squad_response, team_id, team_name, league_id, league_name, season):
    processed_players = []
    if not squad_response:
        return processed_players
    for squad_info in squad_response:
        players = squad_info.get('players', [])
        for player in players:
            player_data = {
                'season': season,
                'league_id': league_id,
                'league_name': league_name,
                'team_id': team_id,
                'team_name': team_name,
                'player_id': player.get('id'),
                'player_name': player.get('name'),
                'player_age': player.get('age'),
                'player_number': player.get('number'),
                'player_position': player.get('position'),
                'player_photo': player.get('photo', '')
            }
            processed_players.append(player_data)
    return processed_players


def download_squad_data():
    all_squad_basic = []
    seasons = list(range(2018, 2024))

    print("Starting Squad Data Download (2018-2023)")
    for league_name, league_id in LEAGUES.items():
        print(f"\n{league_name} (ID: {league_id})")
        for season in seasons:
            print(f"  Season {season}:")
            teams = get_teams_in_league(league_id, season)
            print(f"    Found {len(teams)} teams")

            for i, team in enumerate(teams, 1):
                team_id = team['id']
                team_name = team['name']
                print(f"    [{i}/{len(teams)}] {team_name}...", end=" ")

                squad = get_squad(team_id)
                if squad:
                    processed = process_squad_data(squad, team_id, team_name, league_id, league_name, season)
                    all_squad_basic.extend(processed)
                    print(f" ({len(processed)} players)")
                else:
                    print(" (no data)")
                time.sleep(0.1)

    if all_squad_basic:
        df_squad = pd.DataFrame(all_squad_basic)
        save_to_raw_data("squad_basic", df_squad)

        print(f" Downloaded {len(df_squad)} player-season records.")
    else:
        print(" No squad data collected.")

if __name__ == "__main__":
    download_squad_data()



Starting Squad Data Download (2018-2023)

Premier League (ID: 39)
  Season 2018:
    Found 20 teams
    [1/20] Manchester United...  (34 players)
    [2/20] Newcastle...  (34 players)
    [3/20] Bournemouth...  (27 players)
    [4/20] Fulham...  (27 players)
    [5/20] Huddersfield...  (39 players)
    [6/20] Watford...  (32 players)
    [7/20] Wolves...  (26 players)
    [8/20] Liverpool...  (34 players)
    [9/20] Southampton...  (32 players)
    [10/20] Arsenal...  (30 players)
    [11/20] Cardiff...  (38 players)
    [12/20] Burnley...  (37 players)
    [13/20] Everton...  (30 players)
    [14/20] Leicester...  (29 players)
    [15/20] Tottenham...  (38 players)
    [16/20] West Ham...  (29 players)
    [17/20] Chelsea...  (35 players)
    [18/20] Manchester City...  (32 players)
    [19/20] Brighton...  (33 players)
    [20/20] Crystal Palace...  (30 players)
  Season 2019:
    Found 20 teams
    [1/20] Manchester United...  (34 players)
    [2/20] Newcastle...  (34 players)
    [

####  **6.5 Get Fixtures Of 2024 Season**

In [17]:
def get_fixtures_by_season(league_id, season):
    """Fetch all fixtures for a specific league and season"""
    url = f"{BASE_URL}/fixtures"
    params = {
        'league': league_id,
        'season': season
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        return data['response']
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        return []

def process_fixture_data(fixtures, include_upcoming=False):
    """Process raw fixture data into structured format"""
    processed_finished = []
    processed_upcoming = []
    
    for fixture in fixtures:
        status = fixture['fixture']['status']['short']
        
        # Separate finished matches from upcoming
        is_finished = status in ['FT', 'AET', 'PEN']
        
        match_data = {
            'fixture_id': fixture['fixture']['id'],
            'date': fixture['fixture']['date'],
            'timestamp': fixture['fixture']['timestamp'],
            'league_id': fixture['league']['id'],
            'league_name': fixture['league']['name'],
            'season': fixture['league']['season'],
            'round': fixture['league']['round'],
            'home_team_id': fixture['teams']['home']['id'],
            'home_team_name': fixture['teams']['home']['name'],
            'away_team_id': fixture['teams']['away']['id'],
            'away_team_name': fixture['teams']['away']['name'],
            'venue_name': fixture['fixture']['venue']['name'],
            'venue_city': fixture['fixture']['venue']['city'],
            'referee': fixture['fixture']['referee'],
            'status': status,
            'status_long': fixture['fixture']['status']['long']
        }
        
        if is_finished:
            # Add result data for finished matches
            match_data.update({
                'home_goals': fixture['goals']['home'],
                'away_goals': fixture['goals']['away'],
                'home_score_halftime': fixture['score']['halftime']['home'],
                'away_score_halftime': fixture['score']['halftime']['away'],
                'home_score_fulltime': fixture['score']['fulltime']['home'],
                'away_score_fulltime': fixture['score']['fulltime']['away']
            })
            processed_finished.append(match_data)
        else:
            # Upcoming matches don't have scores yet
            match_data.update({
                'home_goals': None,
                'away_goals': None,
                'home_score_halftime': None,
                'away_score_halftime': None,
                'home_score_fulltime': None,
                'away_score_fulltime': None
            })
            processed_upcoming.append(match_data)
    
    if include_upcoming:
        return processed_finished, processed_upcoming
    return processed_finished, []

def main():
    """Main function to download 2024 fixtures (finished and upcoming)"""
    all_finished_fixtures = []
    all_upcoming_fixtures = []
    season = 2024
    
    print("Starting 2024 Fixtures Download (Finished and Upcoming Matches)")
    print()
    
    for league_name, league_id in LEAGUES.items():
        print(f"\n{league_name} (ID: {league_id})")
        print(f"  Fetching 2024 season data...", end=" ")
        
        fixtures = get_fixtures_by_season(league_id, season)
        
        if fixtures:
            finished, upcoming = process_fixture_data(fixtures, include_upcoming=True)
            all_finished_fixtures.extend(finished)
            all_upcoming_fixtures.extend(upcoming)
            print(f"")
            print(f"    Finished matches: {len(finished)}")
            print(f"    Upcoming matches: {len(upcoming)}")
        else:
            print(" No data")
        
        # Rate limiting
        time.sleep(0.5)
    
    # Save finished 2024 matches (TEST SET)
    if all_finished_fixtures:
        df_finished = pd.DataFrame(all_finished_fixtures)
        saved_file_path = save_to_raw_data("fixtures_2024_Finished", df_finished)
        df_finished.to_csv(saved_file_path, index=False)
        
        print(f"TEST SET: Downloaded {len(df_finished)} finished 2024 matches")
        print(f"Saved to: {saved_file_path}")
        print("\nTest Set Summary:")
        print(f"  Leagues: {df_finished['league_name'].nunique()}")
        print(f"  Teams: {df_finished['home_team_name'].nunique()}")
        print(f"  Date range: {df_finished['date'].min()} to {df_finished['date'].max()}")
        print("\n  Use this file to TEST model's accuracy on unseen 2024 data")
    
    # Save upcoming 2024 matches (PREDICTION SET)
    if all_upcoming_fixtures:
        df_upcoming = pd.DataFrame(all_upcoming_fixtures)
        saved_file_path1 = save_to_raw_data("fixtures_2024_Upcoming", df_upcoming)
        df_upcoming.to_csv(saved_file_path1, index=False)
        
        print(f"PREDICTION SET: Downloaded {len(df_upcoming)} upcoming 2024 matches")
        print(f"Saved to: {saved_file_path1}")
        print("\nPrediction Set Summary:")
        print(f"  Leagues: {df_upcoming['league_name'].nunique()}")
        print(f"  Teams: {df_upcoming['home_team_name'].nunique()}")
        print(f"  Next match: {df_upcoming['date'].min()}")
        print("\n Using this file to make PREDICTIONS with trained model!")
    
    if not all_finished_fixtures and not all_upcoming_fixtures:
        print("\n No data collected!")
    
    # Summary
    print("DOWNLOAD COMPLETE")
    print(f"Total 2024 finished matches: {len(all_finished_fixtures)}")
    print(f"Total 2024 upcoming matches: {len(all_upcoming_fixtures)}")


if __name__ == "__main__":
    main()

Starting 2024 Fixtures Download (Finished and Upcoming Matches)


Premier League (ID: 39)
  Fetching 2024 season data... 
    Finished matches: 380
    Upcoming matches: 0

La Liga (ID: 140)
  Fetching 2024 season data... 
    Finished matches: 380
    Upcoming matches: 0

Serie A (ID: 135)
  Fetching 2024 season data... 
    Finished matches: 380
    Upcoming matches: 0

Bundesliga (ID: 78)
  Fetching 2024 season data... 
    Finished matches: 308
    Upcoming matches: 0

Ligue 1 (ID: 61)
  Fetching 2024 season data... 
    Finished matches: 308
    Upcoming matches: 0

UEFA Champions League (ID: 2)
  Fetching 2024 season data... 
    Finished matches: 279
    Upcoming matches: 0

UEFA Europa League (ID: 3)
  Fetching 2024 season data... 
    Finished matches: 269
    Upcoming matches: 0

UEFA Europa Conference League (ID: 848)
  Fetching 2024 season data... 
    Finished matches: 409
    Upcoming matches: 0

Eredivisie (ID: 88)
  Fetching 2024 season data... 
    Finished matches: 32

####  **6.6 Calculate H2H Matches** 

In [31]:
def calculate_h2h_features(fixtures_df, home_team_id, away_team_id, target_date, num_recent_matches=5):
    """
    Calculate Head-to-Head (H2H) features for a specific fixture.
    """
    # Convert target_date to datetime
    target_date = pd.to_datetime(target_date)

    # Filter historical matches between these two teams before target date
    h2h_matches = fixtures_df[
        (
            ((fixtures_df['home_team_id'] == home_team_id) & (fixtures_df['away_team_id'] == away_team_id)) |
            ((fixtures_df['home_team_id'] == away_team_id) & (fixtures_df['away_team_id'] == home_team_id))
        ) &
        (pd.to_datetime(fixtures_df['date']) < target_date) &
        (fixtures_df['status'].isin(['FT', 'AET', 'PEN']))
    ].copy()

    h2h_matches.sort_values('date', ascending=False, inplace=True)

    # Initialize features
    features = {
        'h2h_total_matches': len(h2h_matches),
        'h2h_home_wins': 0,
        'h2h_draws': 0,
        'h2h_away_wins': 0,
        'h2h_home_win_pct': 0,
        'h2h_draw_pct': 0,
        'h2h_away_win_pct': 0,
        'h2h_avg_home_goals': 0,
        'h2h_avg_away_goals': 0,
        'h2h_avg_total_goals': 0,
        'h2h_btts_pct': 0,
        'h2h_over_2_5_pct': 0,
        'h2h_last_5_home_wins': 0,
        'h2h_last_5_draws': 0,
        'h2h_last_5_away_wins': 0,
        'h2h_days_since_last_meeting': None,
        'h2h_last_result': None
    }

    if len(h2h_matches) == 0:
        return features

    home_goals_list = []
    away_goals_list = []

    # Loop over matches to calculate results
    for _, match in h2h_matches.iterrows():
        if match['home_team_id'] == home_team_id:
            team_goals = match['home_goals']
            opponent_goals = match['away_goals']
        else:
            team_goals = match['away_goals']
            opponent_goals = match['home_goals']

        home_goals_list.append(team_goals)
        away_goals_list.append(opponent_goals)

        if team_goals > opponent_goals:
            features['h2h_home_wins'] += 1
        elif team_goals == opponent_goals:
            features['h2h_draws'] += 1
        else:
            features['h2h_away_wins'] += 1

    total_matches = len(h2h_matches)
    features['h2h_home_win_pct'] = round((features['h2h_home_wins'] / total_matches) * 100, 2)
    features['h2h_draw_pct'] = round((features['h2h_draws'] / total_matches) * 100, 2)
    features['h2h_away_win_pct'] = round((features['h2h_away_wins'] / total_matches) * 100, 2)
    
    features['h2h_avg_home_goals'] = round(np.mean(home_goals_list), 2)
    features['h2h_avg_away_goals'] = round(np.mean(away_goals_list), 2)
    features['h2h_avg_total_goals'] = round(np.mean([h + a for h, a in zip(home_goals_list, away_goals_list)]), 2)
    
    features['h2h_btts_pct'] = round((sum(1 for h, a in zip(home_goals_list, away_goals_list) if h > 0 and a > 0) / total_matches) * 100, 2)
    features['h2h_over_2_5_pct'] = round((sum(1 for h, a in zip(home_goals_list, away_goals_list) if (h + a) > 2.5) / total_matches) * 100, 2)

    # Last N matches
    recent_matches = h2h_matches.head(num_recent_matches)
    for _, match in recent_matches.iterrows():
        if match['home_team_id'] == home_team_id:
            team_goals = match['home_goals']
            opponent_goals = match['away_goals']
        else:
            team_goals = match['away_goals']
            opponent_goals = match['home_goals']

        if team_goals > opponent_goals:
            features['h2h_last_5_home_wins'] += 1
        elif team_goals == opponent_goals:
            features['h2h_last_5_draws'] += 1
        else:
            features['h2h_last_5_away_wins'] += 1

    # Days since last meeting
    features['h2h_days_since_last_meeting'] = (target_date - pd.to_datetime(h2h_matches.iloc[0]['date'])).days

    # Last result
    last_match = h2h_matches.iloc[0]
    if last_match['home_team_id'] == home_team_id:
        last_team_goals = last_match['home_goals']
        last_opponent_goals = last_match['away_goals']
    else:
        last_team_goals = last_match['away_goals']
        last_opponent_goals = last_match['home_goals']

    if last_team_goals > last_opponent_goals:
        features['h2h_last_result'] = 'W'
    elif last_team_goals == last_opponent_goals:
        features['h2h_last_result'] = 'D'
    else:
        features['h2h_last_result'] = 'L'

    return features


def calculate_h2h_for_all_fixtures(fixtures_df, num_recent_matches=5, save_to_file=None):
    """
    Calculate H2H features for all fixtures in the dataset.
    """
    fixtures_df = fixtures_df.sort_values('date').reset_index(drop=True)
    all_features = []

    print(f"Processing {len(fixtures_df)} fixtures...")

    for idx, row in fixtures_df.iterrows():
        if (idx + 1) % 500 == 0:
            print(f"Processed {idx + 1}/{len(fixtures_df)} fixtures")

        h2h = calculate_h2h_features(
            fixtures_df,
            home_team_id=row['home_team_id'],
            away_team_id=row['away_team_id'],
            target_date=row['date'],
            num_recent_matches=num_recent_matches
        )
        all_features.append(h2h)

    h2h_df = pd.DataFrame(all_features)
    result_df = pd.concat([fixtures_df.reset_index(drop=True), h2h_df], axis=1)

    if save_to_file:
        result_df.to_csv(save_to_file, index=False)
        print(f"Saved to {save_to_file}")

    return result_df


if __name__ == "__main__":
    # Load fixtures data
    fixtures_df = pd.read_csv('/Users/vishalchaudhary/Desktop/Football_Project/DOMAIN_APPLICATIONS_PROJECT/data/raw/historical_fixtures_2018_2023_20251101_203250.csv')
    print(f"Loaded {len(fixtures_df)} fixtures")

    # Calculate H2H for all fixtures
    fixtures_with_h2h = calculate_h2h_for_all_fixtures(fixtures_df, num_recent_matches=5)

    # Save the result to raw data folder using your function
    saved_file_path = save_to_raw_data("h2h_matches", fixtures_with_h2h)

    print(f"Sample of H2H features:")
    print(fixtures_with_h2h[['home_team_name', 'away_team_name', 'h2h_total_matches', 
                              'h2h_home_win_pct', 'h2h_avg_total_goals']].head(10))

Loaded 18894 fixtures
Processing 18894 fixtures...
Processed 500/18894 fixtures
Processed 1000/18894 fixtures
Processed 1500/18894 fixtures
Processed 2000/18894 fixtures
Processed 2500/18894 fixtures
Processed 3000/18894 fixtures
Processed 3500/18894 fixtures
Processed 4000/18894 fixtures
Processed 4500/18894 fixtures
Processed 5000/18894 fixtures
Processed 5500/18894 fixtures
Processed 6000/18894 fixtures
Processed 6500/18894 fixtures
Processed 7000/18894 fixtures
Processed 7500/18894 fixtures
Processed 8000/18894 fixtures
Processed 8500/18894 fixtures
Processed 9000/18894 fixtures
Processed 9500/18894 fixtures
Processed 10000/18894 fixtures
Processed 10500/18894 fixtures
Processed 11000/18894 fixtures
Processed 11500/18894 fixtures
Processed 12000/18894 fixtures
Processed 12500/18894 fixtures
Processed 13000/18894 fixtures
Processed 13500/18894 fixtures
Processed 14000/18894 fixtures
Processed 14500/18894 fixtures
Processed 15000/18894 fixtures
Processed 15500/18894 fixtures
Processe

### **6.7 Get Odds Data**

In [ ]:
SEASONS = range(2018, 2023)  
odds_list = []

print("Fetching odds data for seasons 2018-2023...\n")
for season in SEASONS:
    for league_name, league_id in LEAGUES.items():
        url = f"{BASE_URL}/odds"
        params = {
            "league": league_id,
            "season": season,
            "bookmaker": 8  # example bookmaker (Bet365)
        }
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            data = response.json()
            if data.get("response"):
                for match in data["response"]:
                    fixture = match.get("fixture", {})
                    home_team = match["teams"]["home"]["name"]
                    away_team = match["teams"]["away"]["name"]
                    odds_list.append({
                        "season": season,
                        "league": league_name,
                        "fixture_id": fixture.get("id"),
                        "date": fixture.get("date"),
                        "home_team": home_team,
                        "away_team": away_team,
                        "bookmakers": match.get("bookmakers", [])
                    })
                print(f"✓ {league_name} ({season}) - {len(data['response'])} fixtures loaded")
            else:
                print(f"⚠ No odds found for {league_name} ({season})")
        else:
            print(f"✗ Failed for {league_name} ({season}), status: {response.status_code}")
        time.sleep(1)  # brief pause for rate limit


# Convert to DataFrame
df_odds = pd.DataFrame(odds_list)

# Save using your standard data-saving function
save_to_raw_data("odds_data_2018_2023", df_odds)

print(f"\nSaved {len(df_odds)} total odds records to raw data folder as 'odds_data_2018_2023'")

# Print request summary
total_requests = len(LEAGUES) * len(SEASONS)
print(f"\nTotal API requests made: {total_requests}")


Fetching odds data for seasons 2018-2023...

⚠ No odds found for Premier League (2018)
⚠ No odds found for La Liga (2018)
⚠ No odds found for Serie A (2018)
⚠ No odds found for Bundesliga (2018)
⚠ No odds found for Ligue 1 (2018)
⚠ No odds found for UEFA Champions League (2018)
⚠ No odds found for UEFA Europa League (2018)
⚠ No odds found for UEFA Europa Conference League (2018)
⚠ No odds found for Eredivisie (2018)
⚠ No odds found for Primeira Liga (2018)
⚠ No odds found for Premier League (2019)
⚠ No odds found for La Liga (2019)
⚠ No odds found for Serie A (2019)
⚠ No odds found for Bundesliga (2019)
⚠ No odds found for Ligue 1 (2019)
⚠ No odds found for UEFA Champions League (2019)
⚠ No odds found for UEFA Europa League (2019)
⚠ No odds found for UEFA Europa Conference League (2019)
⚠ No odds found for Eredivisie (2019)
⚠ No odds found for Primeira Liga (2019)
⚠ No odds found for Premier League (2020)
⚠ No odds found for La Liga (2020)
⚠ No odds found for Serie A (2020)
⚠ No odds f

### **6.8 Get Injuries Data**

In [8]:

SEASONS = range(2018, 2023)  # 2018–2023 inclusive
injuries_list = []

print("Fetching injuries data for seasons 2018- 2023...\n")
for season in SEASONS:
    for league_name, league_id in LEAGUES.items():
        url = f"{BASE_URL}/injuries"
        params = {
            "league": league_id,
            "season": season
        }
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            data = response.json()
            if data.get("response"):
                for item in data["response"]:
                    player = item.get("player", {})
                    team = item.get("team", {})
                    fixture = item.get("fixture", {})
                    injury = item.get("player", {}).get("reason", None)

                    injuries_list.append({
                        "season": season,
                        "league": league_name,
                        "player_id": player.get("id"),
                        "player_name": player.get("name"),
                        "age": player.get("age"),
                        "team": team.get("name"),
                        "fixture_id": fixture.get("id"),
                        "fixture_date": fixture.get("date"),
                        "injury_reason": injury
                    })
                print(f" {league_name} ({season}) - {len(data['response'])} injury records")
            else:
                print(f" No injury data found for {league_name} ({season})")
        else:
            print(f"✗ Failed for {league_name} ({season}), status: {response.status_code}")
        time.sleep(0.3)  # avoid API rate-limit

# Convert to DataFrame and save
df_injuries = pd.DataFrame(injuries_list)
save_to_raw_data("odds_data_2018_2023", df_injuries)
print(f"\nSaved {len(df_injuries)} total injury records to injuries_data_2018_2023.csv")

# Print request summary
total_requests = len(LEAGUES) * len(SEASONS)
print(f"\nTotal API requests made: {total_requests}")
print(f"Remaining daily allowance (7500/day plan): {7500 - total_requests}")


Fetching injuries data for seasons 2018- 2023...

 No injury data found for Premier League (2018)
 No injury data found for La Liga (2018)
 No injury data found for Serie A (2018)
 No injury data found for Bundesliga (2018)
 No injury data found for Ligue 1 (2018)
 No injury data found for UEFA Champions League (2018)
 No injury data found for UEFA Europa League (2018)
 No injury data found for UEFA Europa Conference League (2018)
 No injury data found for Eredivisie (2018)
 No injury data found for Primeira Liga (2018)
 No injury data found for Premier League (2019)
 No injury data found for La Liga (2019)
 No injury data found for Serie A (2019)
 No injury data found for Bundesliga (2019)
 No injury data found for Ligue 1 (2019)
 No injury data found for UEFA Champions League (2019)
 No injury data found for UEFA Europa League (2019)
 No injury data found for UEFA Europa Conference League (2019)
 No injury data found for Eredivisie (2019)
 No injury data found for Primeira Liga (2019

### **6.9 Get Players Statistics Data**

In [21]:
import requests
import pandas as pd
import time
import os

API_KEY = "b45dd66d18aef1ffb86f5f1df8c83aed"
BASE_URL = "https://v3.football.api-sports.io"
headers = {"x-apisports-key": API_KEY}
LEAGUES = {
    'Premier League': 39,  # 1020 players (51 pages)
    'La Liga': 140,        # 900 players (45 pages)
    'Serie A': 135,        # 1080 players (54 pages)
    'Bundesliga': 78,      # 780 players (39 pages)
    'Ligue 1': 61,         # 860 players (43 pages)
    'UEFA Champions League': 2,  # 2520 players (126 pages)
    'UEFA Europa League': 3,     # 1900 players (95 pages)
    'UEFA Europa Conference League': 848,  # 5200 players (260 pages)
    'Eredivisie': 88,      # 980 players (49 pages)
    'Primeira Liga': 94    # 840 players (42 pages)
}
SEASONS = range(2018, 2024)
SAVE_FILE = "top_500_player_stats_2018_2023.csv"
STATE_FILE = "top_500_player_stats_state.txt"
SAVE_INTERVAL = 500
MAX_PLAYERS_PER_LEAGUE = 500
PLAYERS_PER_PAGE = 20
MAX_PAGES = MAX_PLAYERS_PER_LEAGUE // PLAYERS_PER_PAGE  # 25 pages
player_stats = []
req_count = 36  # 10 prior + 26 for Premier League 2019

def safe_request(url, params):
    for _ in range(3):
        try:
            r = requests.get(url, headers=headers, params=params)
            if r.status_code == 200:
                return r.json()
            print(f"HTTP {r.status_code} for {url} with params {params}")
            time.sleep(2)
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(3)
    return None

def save_partial_data(data, filename):
    if not data:
        return
    df = pd.DataFrame(data)
    if os.path.exists(filename):
        existing = pd.read_csv(filename)
        df = pd.concat([existing, df], ignore_index=True)
    df.to_csv(filename, index=False)
    print(f"Saved partial data ({len(df)} rows) → {filename}")

def save_state(season, league_name, page, req_count):
    with open(STATE_FILE, "w") as f:
        f.write(f"{season},{league_name},{page},{req_count}")

def load_state():
    if not os.path.exists(STATE_FILE):
        return None
    with open(STATE_FILE, "r") as f:
        line = f.read().strip()
        if line:
            parts = line.split(",")
            try:
                if len(parts) == 5:  # Handle old format with collected_players
                    season, league_name, page, req, _ = parts
                    return int(season), league_name, int(page), int(req)
                elif len(parts) == 4:  # Current format
                    season, league_name, page, req = parts
                    return int(season), league_name, int(page), int(req)
                else:
                    print(f"Invalid state file format: {line}. Starting fresh.")
                    return None
            except ValueError as e:
                print(f"Error parsing state file: {e}. Starting fresh.")
                return None
    return None

print("Resuming top 500 player statistics download (2018–2023)...")
state = load_state()
start_season, start_league, start_page, req_count = state if state else (2019, "Premier League", 26, req_count)
resumed = bool(state)

for season in SEASONS:
    if resumed and season < start_season:
        continue
    for league_name, league_id in LEAGUES.items():
        if resumed and league_name != start_league and season == start_season:
            continue
        page = start_page if resumed and season == start_season and league_name == start_league else 1
        league_players = []  # Collect all players for this league/season
        resumed = False
        while len(league_players) < MAX_PLAYERS_PER_LEAGUE and page <= MAX_PAGES:
            url = f"{BASE_URL}/players"
            params = {"league": league_id, "season": season, "page": page}
            data = safe_request(url, params)
            req_count += 1

            if req_count % SAVE_INTERVAL == 0:
                save_partial_data(player_stats, SAVE_FILE)
                save_state(season, league_name, page, req_count)
                player_stats = []

            if not data or not data.get("response"):
                print(f"No more data for {league_name} ({season}) at page {page}")
                break

            for p in data["response"]:
                if len(league_players) >= MAX_PLAYERS_PER_LEAGUE:
                    break
                player = p.get("player", {})
                stats = p.get("statistics", [{}])[0]
                team = stats.get("team", {})
                games = stats.get("games", {})
                goals = stats.get("goals", {})
                passes = stats.get("passes", {})
                shots = stats.get("shots", {})
                tackles = stats.get("tackles", {})
                duels = stats.get("duels", {})
                dribbles = stats.get("dribbles", {})
                fouls = stats.get("fouls", {})
                rating = float(games.get("rating", 0) or 0)
                league_players.append({
                    "season": season,
                    "league": league_name,
                    "player_id": player.get("id"),
                    "player_name": player.get("name"),
                    "age": player.get("age"),
                    "nationality": player.get("nationality"),
                    "team": team.get("name"),
                    "position": games.get("position"),
                    "appearances": games.get("appearances"),
                    "minutes": games.get("minutes"),
                    "rating": rating,
                    "goals_total": goals.get("total"),
                    "assists": goals.get("assists"),
                    "shots_total": shots.get("total"),
                    "shots_on": shots.get("on"),
                    "passes_total": passes.get("total"),
                    "key_passes": passes.get("key"),
                    "tackles_total": tackles.get("total"),
                    "interceptions": tackles.get("interceptions"),
                    "duels_total": duels.get("total"),
                    "duels_won": duels.get("won"),
                    "dribbles_success": dribbles.get("success"),
                    "fouls_committed": fouls.get("committed")
                })

            print(f"{league_name} ({season}) page {page} — {len(data['response'])} players (Total: {len(league_players)})")
            
            if data.get("paging", {}).get("current") == data.get("paging", {}).get("total") or not data["response"]:
                break
            page += 1
            time.sleep(0.7)

        # Sort players by rating (descending) and take top 500
        league_players = sorted(league_players, key=lambda x: x["rating"], reverse=True)[:MAX_PLAYERS_PER_LEAGUE]
        player_stats.extend(league_players)
        print(f"Selected top {len(league_players)} players for {league_name} ({season})")
        
        # Save after each league/season
        save_partial_data(player_stats, SAVE_FILE)
        save_state(season, league_name, page, req_count)
        player_stats = []

# Final save and cleanup
save_partial_data(player_stats, SAVE_FILE)
if os.path.exists(STATE_FILE):
    os.remove(STATE_FILE)

print(f"\nCompleted. Total requests: {req_count}")
print(f"Remaining daily limit: {7500 - req_count}")

Resuming top 500 player statistics download (2018–2023)...
Selected top 0 players for Ligue 1 (2019)
UEFA Champions League (2019) page 1 — 20 players (Total: 20)
UEFA Champions League (2019) page 2 — 20 players (Total: 40)
UEFA Champions League (2019) page 3 — 20 players (Total: 60)
UEFA Champions League (2019) page 4 — 20 players (Total: 80)
UEFA Champions League (2019) page 5 — 20 players (Total: 100)
UEFA Champions League (2019) page 6 — 20 players (Total: 120)
UEFA Champions League (2019) page 7 — 20 players (Total: 140)
UEFA Champions League (2019) page 8 — 20 players (Total: 160)
UEFA Champions League (2019) page 9 — 20 players (Total: 180)
UEFA Champions League (2019) page 10 — 20 players (Total: 200)
UEFA Champions League (2019) page 11 — 20 players (Total: 220)
UEFA Champions League (2019) page 12 — 20 players (Total: 240)
UEFA Champions League (2019) page 13 — 20 players (Total: 260)
UEFA Champions League (2019) page 14 — 20 players (Total: 280)
UEFA Champions League (2019) p

/var/folders/gw/ck0g9c4n03ld5qbdn_p8skp00000gn/T/ipykernel_18868/2547607307.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([existing, df], ignore_index=True)


UEFA Europa League (2019) page 1 — 20 players (Total: 20)
UEFA Europa League (2019) page 2 — 20 players (Total: 40)
UEFA Europa League (2019) page 3 — 20 players (Total: 60)
UEFA Europa League (2019) page 4 — 20 players (Total: 80)
UEFA Europa League (2019) page 5 — 20 players (Total: 100)
UEFA Europa League (2019) page 6 — 20 players (Total: 120)
UEFA Europa League (2019) page 7 — 20 players (Total: 140)
UEFA Europa League (2019) page 8 — 20 players (Total: 160)
UEFA Europa League (2019) page 9 — 20 players (Total: 180)
UEFA Europa League (2019) page 10 — 20 players (Total: 200)
UEFA Europa League (2019) page 11 — 20 players (Total: 220)
UEFA Europa League (2019) page 12 — 20 players (Total: 240)
UEFA Europa League (2019) page 13 — 20 players (Total: 260)
UEFA Europa League (2019) page 14 — 20 players (Total: 280)
UEFA Europa League (2019) page 15 — 20 players (Total: 300)
UEFA Europa League (2019) page 16 — 20 players (Total: 320)
UEFA Europa League (2019) page 17 — 20 players (Total

In [14]:
import requests
import pandas as pd
import time
from pathlib import Path

API_KEY = "b45dd66d18aef1ffb86f5f1df8c83aed"
BASE_URL = "https://v3.football.api-sports.io"
headers = {"x-apisports-key": API_KEY}

output_file = Path('fixture_statistics_complete.csv')

print("OPTIMIZED FIXTURE STATISTICS COLLECTION")
print(f"Loading existing fixtures from fixtures_with_target.csv...")

df_fixtures = pd.read_csv('/Users/vishalchaudhary/Desktop/Football_Project/DOMAIN_APPLICATIONS_PROJECT/data/processed/fixtures_with_target.csv')

print(f"Found {len(df_fixtures):,} fixtures to process")

collected_data = []
request_count = 0

if output_file.exists():
    existing_df = pd.read_csv(output_file)
    collected_data = existing_df.to_dict('records')
    existing_fixtures = set([str(row['fixture_id']) for row in collected_data])
    print(f"Resuming from {len(collected_data)} existing rows\n")
else:
    existing_fixtures = set()

start_time = time.time()

for idx, row in df_fixtures.iterrows():
    fixture_id = row['fixture_id']
    
    if str(fixture_id) in existing_fixtures:
        continue
    
    try:
        stats_url = f"{BASE_URL}/fixtures/statistics"
        stats_params = {"fixture": fixture_id}
        
        stats_response = requests.get(stats_url, headers=headers, params=stats_params, timeout=10)
        request_count += 1
        
        if stats_response.status_code == 200:
            stats_data = stats_response.json()
            
            if 'response' in stats_data and len(stats_data['response']) >= 2:
                home_stats = {stat['type']: stat['value'] for stat in stats_data['response'][0]['statistics']}
                away_stats = {stat['type']: stat['value'] for stat in stats_data['response'][1]['statistics']}
                
                data_row = {
                    'fixture_id': fixture_id,
                    'league_id': row.get('league_id'),
                    'league_name': row.get('league_name'),
                    'season': row.get('season'),
                    'date': row.get('date'),
                    'home_team_id': row.get('home_team_id'),
                    'home_team_name': row.get('home_team_name'),
                    'away_team_id': row.get('away_team_id'),
                    'away_team_name': row.get('away_team_name'),
                    'home_shots_on_goal': home_stats.get('Shots on Goal'),
                    'away_shots_on_goal': away_stats.get('Shots on Goal'),
                    'home_total_shots': home_stats.get('Total Shots'),
                    'away_total_shots': away_stats.get('Total Shots'),
                    'home_fouls': home_stats.get('Fouls'),
                    'away_fouls': away_stats.get('Fouls'),
                    'home_corners': home_stats.get('Corner Kicks'),
                    'away_corners': away_stats.get('Corner Kicks'),
                    'home_possession': home_stats.get('Ball Possession'),
                    'away_possession': away_stats.get('Ball Possession'),
                    'home_yellow_cards': home_stats.get('Yellow Cards'),
                    'away_yellow_cards': away_stats.get('Yellow Cards'),
                    'home_passes_accurate': home_stats.get('Passes accurate'),
                    'away_passes_accurate': away_stats.get('Passes accurate'),
                    'home_passes_percent': home_stats.get('Passes %'),
                    'away_passes_percent': away_stats.get('Passes %')
                }
                
                collected_data.append(data_row)
                existing_fixtures.add(str(fixture_id))
                
                if len(collected_data) % 500 == 0:
                    df_temp = pd.DataFrame(collected_data)
                    df_temp.to_csv(output_file, index=False)
                    elapsed = time.time() - start_time
                    rate = len(collected_data) / elapsed * 60
                    remaining = len(df_fixtures) - len(collected_data)
                    eta_hours = remaining / rate / 60
                    print(f"Saved: {len(collected_data)} | Requests: {request_count} | Rate: {rate:.1f}/min | ETA: {eta_hours:.1f}h")
        
        time.sleep(0.05)
        
        if request_count >= 7400:
            print(f"\nAPI LIMIT REACHED ({request_count}/7500)")
            df_final = pd.DataFrame(collected_data)
            df_final.to_csv(output_file, index=False)
            print(f"Saved {len(collected_data)} fixtures. Resume tomorrow!")
            break
            
    except Exception as e:
        print(f"Error on fixture {fixture_id}: {e}")
        time.sleep(0.2)
        continue

df_final = pd.DataFrame(collected_data)
df_final.to_csv(output_file, index=False)

print(f"\nCOLLECTED: {len(collected_data):,} fixtures")
print(f"REQUESTS: {request_count:,}")
print(f"TIME: {(time.time()-start_time)/60:.1f} min")

OPTIMIZED FIXTURE STATISTICS COLLECTION
Loading existing fixtures from fixtures_with_target.csv...
Found 18,894 fixtures to process
Resuming from 17488 existing rows

Saved: 17500 | Requests: 1199 | Rate: 5693.7/min | ETA: 0.0h

COLLECTED: 17,628 fixtures
REQUESTS: 1,406
TIME: 3.6 min


In [17]:
# Leagues and their IDs
LEAGUES = {
    'Premier League': 39,
    'La Liga': 140,
    'Serie A': 135,
    'Bundesliga': 78,
    'Ligue 1': 61,
    'UEFA Champions League': 2,
    'UEFA Europa League': 3,
    'UEFA Europa Conference League': 848,
    'Eredivisie': 88,
    'Primeira Liga': 94
}

# Seasons
SEASONS = ['2018', '2019', '2020', '2021', '2022', '2023']

# Average matches per season per league (approximate)
MATCHES_PER_SEASON = {
    'Premier League': 380,
    'La Liga': 380,
    'Serie A': 380,
    'Bundesliga': 306,
    'Ligue 1': 380,
    'UEFA Champions League': 125,  # Group + Knockout
    'UEFA Europa League': 205,  # More teams
    'UEFA Europa Conference League': 141,  # Started 2021
    'Eredivisie': 306,
    'Primeira Liga': 306
}

print("API REQUEST CALCULATIONS FOR 2018-2023")

# 1. LINEUPS - 1 request per match
print("\n1. LINEUPS (/fixtures/lineups)")
lineup_requests = 0
for league, matches in MATCHES_PER_SEASON.items():
    if league == 'UEFA Europa Conference League':
        seasons = 3  # Started in 2021
    else:
        seasons = 6
    league_total = matches * seasons
    lineup_requests += league_total
    print(f"{league:35} {matches:4} matches/season × {seasons} seasons = {league_total:5} requests")

print(f"\n{'TOTAL LINEUP REQUESTS':35} = {lineup_requests:5} requests")

# 2. COACHES - 1 request per team per season
print("\n\n2. COACHES (/coachs)")
print("Note: Need to get team IDs first, then query each coach")

TEAMS_PER_LEAGUE = {
    'Premier League': 20,
    'La Liga': 20,
    'Serie A': 20,
    'Bundesliga': 18,
    'Ligue 1': 20,
    'UEFA Champions League': 32,  # Varies by season
    'UEFA Europa League': 48,  # Varies by season
    'UEFA Europa Conference League': 32,  # Varies by season
    'Eredivisie': 18,
    'Primeira Liga': 18
}

coach_requests = 0
for league, teams in TEAMS_PER_LEAGUE.items():
    if league == 'UEFA Europa Conference League':
        seasons = 3
    else:
        seasons = 6
    league_total = teams * seasons
    coach_requests += league_total
    print(f"{league:35} {teams:3} teams × {seasons} seasons = {league_total:4} requests")

print(f"\n{'TOTAL COACH REQUESTS':35} = {coach_requests:5} requests")

# 3. EVENTS - 1 request per match
print("\n\n3. EVENTS (/fixtures/events)")
event_requests = 0
for league, matches in MATCHES_PER_SEASON.items():
    if league == 'UEFA Europa Conference League':
        seasons = 3
    else:
        seasons = 6
    league_total = matches * seasons
    event_requests += league_total
    print(f"{league:35} {matches:4} matches/season × {seasons} seasons = {league_total:5} requests")

print(f"\n{'TOTAL EVENT REQUESTS':35} = {event_requests:5} requests")

# GRAND TOTAL
print("SUMMARY")
print(f"Lineups:  {lineup_requests:6,} requests")
print(f"Coaches:  {coach_requests:6,} requests")
print(f"Events:   {event_requests:6,} requests")
print(f"{'-' * 40}")
print(f"TOTAL:    {lineup_requests + coach_requests + event_requests:6,} requests")

# Rate limit calculations
print("TIME ESTIMATES (at 400 requests/minute)")
total_requests = lineup_requests + coach_requests + event_requests
minutes = total_requests / 400
hours = minutes / 60

print(f"Total time needed: {minutes:.1f} minutes ({hours:.2f} hours)")
print(f"Lineups only: {lineup_requests/400:.1f} minutes")
print(f"Coaches only: {coach_requests/400:.1f} minutes")
print(f"Events only: {event_requests/400:.1f} minutes")

# Cost analysis (if applicable)
print("API SUBSCRIPTION CHECK")
print(f"Total requests needed: {total_requests:,}")
print("Your API-Football Pro subscription should handle this.")
print("Just make sure to implement proper rate limiting and error handling!")

API REQUEST CALCULATIONS FOR 2018-2023

1. LINEUPS (/fixtures/lineups)
Premier League                       380 matches/season × 6 seasons =  2280 requests
La Liga                              380 matches/season × 6 seasons =  2280 requests
Serie A                              380 matches/season × 6 seasons =  2280 requests
Bundesliga                           306 matches/season × 6 seasons =  1836 requests
Ligue 1                              380 matches/season × 6 seasons =  2280 requests
UEFA Champions League                125 matches/season × 6 seasons =   750 requests
UEFA Europa League                   205 matches/season × 6 seasons =  1230 requests
UEFA Europa Conference League        141 matches/season × 3 seasons =   423 requests
Eredivisie                           306 matches/season × 6 seasons =  1836 requests
Primeira Liga                        306 matches/season × 6 seasons =  1836 requests

TOTAL LINEUP REQUESTS               = 17031 requests


2. COACHES (/coachs)
No

In [1]:
import requests
import pandas as pd
import time
from pathlib import Path

API_KEY = "b45dd66d18aef1ffb86f5f1df8c83aed"
BASE_URL = "https://v3.football.api-sports.io"
headers = {"x-apisports-key": API_KEY}

output_file = Path('lineups_complete.csv')

print("OPTIMIZED LINEUPS DATA COLLECTION")
print(f"Loading existing fixtures from fixtures_with_target.csv...")

df_fixtures = pd.read_csv('/Users/vishalchaudhary/Desktop/Football_Project/DOMAIN_APPLICATIONS_PROJECT/data/processed/fixtures_with_target.csv')

print(f"Found {len(df_fixtures):,} fixtures to process")

collected_data = []
request_count = 0

if output_file.exists():
    existing_df = pd.read_csv(output_file)
    collected_data = existing_df.to_dict('records')
    existing_fixtures = set([str(row['fixture_id']) for row in collected_data])
    print(f"Resuming from {len(collected_data)} existing rows\n")
else:
    existing_fixtures = set()

start_time = time.time()

for idx, row in df_fixtures.iterrows():
    fixture_id = row['fixture_id']
    
    if str(fixture_id) in existing_fixtures:
        continue
    
    try:
        lineups_url = f"{BASE_URL}/fixtures/lineups"
        lineups_params = {"fixture": fixture_id}
        
        lineups_response = requests.get(lineups_url, headers=headers, params=lineups_params, timeout=10)
        request_count += 1
        
        if lineups_response.status_code == 200:
            lineups_data = lineups_response.json()
            
            if 'response' in lineups_data and len(lineups_data['response']) >= 2:
                home_lineup = lineups_data['response'][0]
                away_lineup = lineups_data['response'][1]
                
                # Extract home team data
                home_team = home_lineup.get('team', {})
                home_coach = home_lineup.get('coach', {})
                home_formation = home_lineup.get('formation', None)
                home_startXI = home_lineup.get('startXI', [])
                home_substitutes = home_lineup.get('substitutes', [])
                
                # Extract away team data
                away_team = away_lineup.get('team', {})
                away_coach = away_lineup.get('coach', {})
                away_formation = away_lineup.get('formation', None)
                away_startXI = away_lineup.get('startXI', [])
                away_substitutes = away_lineup.get('substitutes', [])
                
                # Count players
                home_startXI_count = len(home_startXI)
                away_startXI_count = len(away_startXI)
                home_subs_count = len(home_substitutes)
                away_subs_count = len(away_substitutes)
                
                data_row = {
                    'fixture_id': fixture_id,
                    'league_id': row.get('league_id'),
                    'league_name': row.get('league_name'),
                    'season': row.get('season'),
                    'date': row.get('date'),
                    'home_team_id': row.get('home_team_id'),
                    'home_team_name': row.get('home_team_name'),
                    'away_team_id': row.get('away_team_id'),
                    'away_team_name': row.get('away_team_name'),
                    
                    # Home team lineup info
                    'home_formation': home_formation,
                    'home_coach_id': home_coach.get('id'),
                    'home_coach_name': home_coach.get('name'),
                    'home_startXI_count': home_startXI_count,
                    'home_subs_count': home_subs_count,
                    
                    # Away team lineup info
                    'away_formation': away_formation,
                    'away_coach_id': away_coach.get('id'),
                    'away_coach_name': away_coach.get('name'),
                    'away_startXI_count': away_startXI_count,
                    'away_subs_count': away_subs_count
                }
                
                collected_data.append(data_row)
                existing_fixtures.add(str(fixture_id))
                
                if len(collected_data) % 100 == 0:
                    df_temp = pd.DataFrame(collected_data)
                    df_temp.to_csv(output_file, index=False)
                    elapsed = time.time() - start_time
                    rate = request_count / elapsed * 60
                    remaining = len(df_fixtures) - len(collected_data)
                    eta_minutes = remaining / rate
                    print(f"Saved: {len(collected_data)} | Requests: {request_count} | Rate: {rate:.1f}/min | ETA: {eta_minutes:.1f}m")
        
        time.sleep(0.05)
        
        if request_count >= 7400:
            print(f"\nAPI LIMIT REACHED ({request_count}/7500)")
            df_final = pd.DataFrame(collected_data)
            df_final.to_csv(output_file, index=False)
            print(f"Saved {len(collected_data)} fixtures. Resume tomorrow!")
            break
            
    except Exception as e:
        print(f"Error on fixture {fixture_id}: {e}")
        time.sleep(0.2)
        continue

df_final = pd.DataFrame(collected_data)
df_final.to_csv(output_file, index=False)

elapsed_total = time.time() - start_time

print("COLLECTION COMPLETE")
print(f"COLLECTED: {len(collected_data):,} fixtures")
print(f"REQUESTS: {request_count:,}")
print(f"TIME: {elapsed_total/60:.1f} minutes")
print(f"RATE: {request_count/elapsed_total*60:.1f} requests/min")
print(f"\nData saved to: {output_file}")

# Show summary if data collected
if len(collected_data) > 0:
    df_summary = pd.DataFrame(collected_data)
    
    print("SUMMARY BY LEAGUE")
    print(df_summary.groupby('league_name')['fixture_id'].count().sort_values(ascending=False))
    
    print("FORMATION DISTRIBUTION")
    print("Home formations:")
    print(df_summary['home_formation'].value_counts().head(10))
    print("\nAway formations:")
    print(df_summary['away_formation'].value_counts().head(10))

OPTIMIZED LINEUPS DATA COLLECTION
Loading existing fixtures from fixtures_with_target.csv...
Found 18,894 fixtures to process
Resuming from 13700 existing rows

Saved: 13800 | Requests: 100 | Rate: 365.4/min | ETA: 13.9m
Saved: 13900 | Requests: 201 | Rate: 352.5/min | ETA: 14.2m
Saved: 14000 | Requests: 302 | Rate: 363.0/min | ETA: 13.5m
Saved: 14100 | Requests: 402 | Rate: 373.2/min | ETA: 12.8m
Saved: 14200 | Requests: 599 | Rate: 384.9/min | ETA: 12.2m
Saved: 14300 | Requests: 700 | Rate: 381.5/min | ETA: 12.0m
Saved: 14400 | Requests: 800 | Rate: 379.6/min | ETA: 11.8m
Saved: 14500 | Requests: 974 | Rate: 378.2/min | ETA: 11.6m
Saved: 14600 | Requests: 1074 | Rate: 377.2/min | ETA: 11.4m
Saved: 14700 | Requests: 1174 | Rate: 376.0/min | ETA: 11.2m
Saved: 14800 | Requests: 1337 | Rate: 372.4/min | ETA: 11.0m
Saved: 14900 | Requests: 1437 | Rate: 368.6/min | ETA: 10.8m
Saved: 15000 | Requests: 1537 | Rate: 369.8/min | ETA: 10.5m
Saved: 15100 | Requests: 1695 | Rate: 370.5/min | ETA: